# Tugas Besar II4013 Data Analytics
## **OSEMN** 
Proyek ini disusun dengan OSEMN Framework: *Obtain, Scrub, Explore, Model, dan iNterpret*. Framework ini menjadi dasar alur kerja teknis, struktur laporan, struktur notebook script, dan urutan presentasi.  

**Anggota Kelompok:**

| Nama | NIM |
|:----:|:---:|
| Bryan P. Hutagalung | 18222130 |
| Naila Selvira Budiana | 18223018 |
| Florecita Natawirya | 18223040 |
| Fhatika Adhalisman Ryanjani | 18223068 |
| Muhammad Refino Ramadhan | 18223070 |

# **1. Obtain**

## *Data Engineering* / 18223040

#### 1. Scope Obtain

Bagian ini menjadi bukti proses **Data Engineering** pada tahap **Obtain**. Fokusnya adalah mengambil data kemiskinan dan kerentanan sosial Jawa Barat langsung dari API Open Data Jabar, menyimpan hasil mentah secara reproducible, dan menyediakan orchestration scaffold agar pipeline bisa dijalankan ulang saat sumber data diperbarui.

Notebook ini tidak mengubah script ekstraksi milik data engineer. Sel di bawah hanya membaca atau memanggil `scripts/extract_api.py` dan `scripts/orchestrate_prefect.py` sebagai proof proses.


#### Checklist Data Engineering

| Kriteria Obtain / Data Engineering | Implementasi di repository | Bukti di notebook |
|---|---|---|
| Mengakses data dari sumber eksternal | Open Data Jabar via dokumen OpenAPI dan endpoint record aktual | Katalog dataset dan endpoint API |
| Automasi extraction | `scripts/extract_api.py` mengambil semua record dengan pagination `limit` dan `skip` | Sel opsi run extraction |
| Penyimpanan raw data | JSON dan CSV mentah disimpan di `data/raw/`, termasuk versi timestamp dan file latest | Audit file raw |
| Logging dan manifest | `logs/ingest.log` dan `data/manifest/ingest_manifest.csv` mencatat status, source URL, checksum | Ringkasan manifest |
| Reproducible pipeline | Script dapat dijalankan ulang; checksum mencegah duplikasi ingest yang sama | Status manifest dan idempotensi |
| Orkestrasi | Scaffold Prefect memanggil script extraction | Validasi isi `orchestrate_prefect.py` |


In [33]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "extract_api.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts import extract_api as extract_module

RAW_DIR = PROJECT_ROOT / "data" / "raw"
MANIFEST_PATH = PROJECT_ROOT / "data" / "manifest" / "ingest_manifest.csv"
EXTRACT_SCRIPT = PROJECT_ROOT / "scripts" / "extract_api.py"
ORCHESTRATE_SCRIPT = PROJECT_ROOT / "scripts" / "orchestrate_prefect.py"

print(f"Project root     : {PROJECT_ROOT}")
print(f"Extract script   : {EXTRACT_SCRIPT.relative_to(PROJECT_ROOT)}")
print(f"Orchestrator     : {ORCHESTRATE_SCRIPT.relative_to(PROJECT_ROOT)}")
print(f"Raw data folder  : {RAW_DIR.relative_to(PROJECT_ROOT)}")
print(f"Manifest         : {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")


Project root     : /home/nathangalung/documents/kuliah/sem8/datnal/data-analytics-tubes
Extract script   : scripts/extract_api.py
Orchestrator     : scripts/orchestrate_prefect.py
Raw data folder  : data/raw
Manifest         : data/manifest/ingest_manifest.csv


#### 2. Katalog Dataset Obtain

Dataset yang dipakai berasal dari Open Data Jabar dan dipilih agar sesuai dengan fokus analytics: tren kemiskinan, kerentanan sosial, dan prioritas intervensi wilayah di Jawa Barat. Lima dataset ini nantinya menjadi input untuk tahap Scrub/preprocessing.


In [34]:
dataset_notes = {
    "raw_garis_kemiskinan": {
        "kelompok": "Kemiskinan",
        "indikator": "Garis kemiskinan",
        "alasan": "Mengukur batas minimum pengeluaran penduduk miskin per wilayah.",
    },
    "raw_persentase_miskin": {
        "kelompok": "Kemiskinan",
        "indikator": "Persentase penduduk miskin",
        "alasan": "Menjadi indikator utama tren kemiskinan kabupaten/kota.",
    },
    "raw_keparahan_kemiskinan": {
        "kelompok": "Kemiskinan",
        "indikator": "Indeks keparahan kemiskinan",
        "alasan": "Menunjukkan kedalaman masalah pada kelompok miskin.",
    },
    "raw_ipm_sp2010": {
        "kelompok": "Kerentanan sosial",
        "indikator": "Indeks Pembangunan Manusia SP2010",
        "alasan": "Mewakili kualitas pembangunan manusia dan kapasitas sosial wilayah.",
    },
    "raw_pengangguran_terbuka": {
        "kelompok": "Kerentanan sosial",
        "indikator": "Tingkat pengangguran terbuka",
        "alasan": "Mewakili tekanan sosial-ekonomi yang dapat memperbesar kerentanan.",
    },
}

def latest_raw_payload(logical_name):
    json_path = RAW_DIR / f"{logical_name}.json"
    if not json_path.exists():
        return {}
    return json.loads(json_path.read_text(encoding="utf-8"))

def summarize_latest_csv(logical_name):
    csv_path = RAW_DIR / f"{logical_name}.csv"
    if not csv_path.exists():
        return {"raw_file": None, "rows": 0, "columns": 0, "tahun_min": None, "tahun_max": None}
    df = pd.read_csv(csv_path)
    tahun = pd.to_numeric(df.get("tahun"), errors="coerce") if "tahun" in df.columns else pd.Series(dtype="float64")
    return {
        "raw_file": csv_path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "tahun_min": int(tahun.min()) if tahun.notna().any() else None,
        "tahun_max": int(tahun.max()) if tahun.notna().any() else None,
    }

catalog_rows = []
for logical_name, doc_url in extract_module.api_docs.items():
    payload = latest_raw_payload(logical_name)
    row = {
        "logical_name": logical_name,
        "kelompok": dataset_notes[logical_name]["kelompok"],
        "indikator": dataset_notes[logical_name]["indikator"],
        "alasan_pemilihan": dataset_notes[logical_name]["alasan"],
        "source_doc_url": doc_url,
        "source_data_url": payload.get("source_data_url"),
    }
    row.update(summarize_latest_csv(logical_name))
    catalog_rows.append(row)

catalog_df = pd.DataFrame(catalog_rows)
catalog_df

,logical_name,kelompok,indikator,alasan_pemilihan,source_doc_url,source_data_url,raw_file,rows,columns,tahun_min,tahun_max
0,raw_garis_kemiskinan,Kemiskinan,Garis kemiskinan,Mengukur batas minimum pengeluaran penduduk mi...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_garis_kemiskinan.csv,594,8,2004,2025
1,raw_persentase_miskin,Kemiskinan,Persentase penduduk miskin,Menjadi indikator utama tren kemiskinan kabupa...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_persentase_miskin.csv,405,8,2010,2024
2,raw_keparahan_kemiskinan,Kemiskinan,Indeks keparahan kemiskinan,Menunjukkan kedalaman masalah pada kelompok mi...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_keparahan_kemiskinan.csv,567,8,2004,2024
3,raw_ipm_sp2010,Kerentanan sosial,Indeks Pembangunan Manusia SP2010,Mewakili kualitas pembangunan manusia dan kapa...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_ipm_sp2010.csv,405,8,2010,2024
4,raw_pengangguran_terbuka,Kerentanan sosial,Tingkat pengangguran terbuka,Mewakili tekanan sosial-ekonomi yang dapat mem...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_pengangguran_terbuka.csv,424,8,2007,2024


#### 3. Cara Akses Data API

Script extraction memakai pendekatan API-first. URL yang didefinisikan di script adalah dokumen OpenAPI; dari dokumen tersebut script mencari endpoint data aktual, lalu mengambil semua record dengan pagination. Output JSON raw juga menyimpan `source_doc_url`, `source_data_url`, metadata jumlah baris, limit, dan jumlah halaman yang berhasil diambil.


In [35]:
endpoint_rows = []
for logical_name in extract_module.api_docs:
    payload = latest_raw_payload(logical_name)
    metadata = payload.get("metadata", {})
    endpoint_rows.append({
        "logical_name": logical_name,
        "source_doc_url": payload.get("source_doc_url", extract_module.api_docs[logical_name]),
        "source_data_url": payload.get("source_data_url"),
        "rows_from_json": metadata.get("rows"),
        "pages": metadata.get("pages"),
        "limit": metadata.get("limit"),
        "ingest_time_utc": payload.get("ingest_time_utc"),
    })

endpoint_audit_df = pd.DataFrame(endpoint_rows)
endpoint_audit_df

,logical_name,source_doc_url,source_data_url,rows_from_json,pages,limit,ingest_time_utc
0,raw_garis_kemiskinan,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,594,1,5000,2026-06-05T15:50:40.827657Z
1,raw_persentase_miskin,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,405,1,5000,2026-06-05T15:50:41.560361Z
2,raw_keparahan_kemiskinan,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,567,1,5000,2026-06-05T15:50:42.262887Z
3,raw_ipm_sp2010,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,405,1,5000,2026-06-05T15:50:42.985793Z
4,raw_pengangguran_terbuka,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,424,1,5000,2026-06-05T15:50:43.718883Z


#### 4. Memanggil Script Extraction

Sel ini adalah titik panggil ke `scripts/extract_api.py`. Untuk mode presentasi, `RUN_EXTRACT_PIPELINE` dibuat `False` agar notebook tidak otomatis melakukan request ulang ke API setiap kali dijalankan. Jika ingin memperbarui data dari Open Data Jabar, ubah nilainya menjadi `True`, lalu jalankan sel ini.


In [36]:
RUN_EXTRACT_PIPELINE = False

command = [sys.executable, str(EXTRACT_SCRIPT)]
print("Command extraction:", " ".join(command))

if RUN_EXTRACT_PIPELINE:
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Extraction gagal dengan return code {result.returncode}")
else:
    print("Mode presentasi: extraction tidak dijalankan ulang. Notebook memakai raw data latest yang sudah tersedia.")

Command extraction: /home/nathangalung/documents/kuliah/sem8/datnal/data-analytics-tubes/.venv/bin/python /home/nathangalung/documents/kuliah/sem8/datnal/data-analytics-tubes/scripts/extract_api.py
Mode presentasi: extraction tidak dijalankan ulang. Notebook memakai raw data latest yang sudah tersedia.


#### 5. Audit Output Raw Data

Audit ini mengecek apakah script extraction sudah menghasilkan pasangan file JSON dan CSV untuk setiap dataset, baik versi latest maupun versi timestamp. File timestamp berguna untuk tracking perubahan data dari waktu ke waktu, sedangkan file latest memudahkan tahap preprocessing membaca data terbaru.


In [37]:
raw_audit_rows = []
for logical_name in extract_module.api_docs:
    latest_csv = RAW_DIR / f"{logical_name}.csv"
    latest_json = RAW_DIR / f"{logical_name}.json"
    versioned_csv = sorted(RAW_DIR.glob(f"{logical_name}_*.csv"))
    versioned_json = sorted(RAW_DIR.glob(f"{logical_name}_*.json"))
    summary = summarize_latest_csv(logical_name)
    raw_audit_rows.append({
        "logical_name": logical_name,
        "latest_csv_exists": latest_csv.exists(),
        "latest_json_exists": latest_json.exists(),
        "versioned_csv_count": len(versioned_csv),
        "versioned_json_count": len(versioned_json),
        "rows": summary["rows"],
        "columns": summary["columns"],
        "tahun_min": summary["tahun_min"],
        "tahun_max": summary["tahun_max"],
    })

raw_audit_df = pd.DataFrame(raw_audit_rows)
raw_audit_df

,logical_name,latest_csv_exists,latest_json_exists,versioned_csv_count,versioned_json_count,rows,columns,tahun_min,tahun_max
0,raw_garis_kemiskinan,True,True,0,0,594,8,2004,2025
1,raw_persentase_miskin,True,True,0,0,405,8,2010,2024
2,raw_keparahan_kemiskinan,True,True,0,0,567,8,2004,2024
3,raw_ipm_sp2010,True,True,0,0,405,8,2010,2024
4,raw_pengangguran_terbuka,True,True,0,0,424,8,2007,2024


#### 6. Manifest, Checksum, dan Idempotensi

Manifest mencatat riwayat ingest. Kolom `checksum` dipakai untuk mendeteksi apakah data yang baru diambil sama dengan data yang sudah pernah disimpan. Jika checksum sama, script menandai proses sebagai `skipped`, sehingga pipeline bisa dijalankan ulang tanpa menumpuk file duplikat yang isinya sama.


In [38]:
manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df["ingest_time"] = pd.to_datetime(manifest_df["ingest_time"], errors="coerce")
manifest_latest_df = (
    manifest_df.sort_values("ingest_time")
    .groupby("logical_name", as_index=False)
    .tail(1)
    .sort_values("logical_name")
)

status_summary_df = (
    manifest_df.groupby(["logical_name", "status"])
    .size()
    .reset_index(name="jumlah_event")
    .sort_values(["logical_name", "status"])
)

display(status_summary_df)
manifest_latest_df[["ingest_time", "logical_name", "stored_filename", "source_url", "status", "notes"]]

,logical_name,status,jumlah_event
0,raw_garis_kemiskinan,failed,1
1,raw_garis_kemiskinan,skipped,5
2,raw_garis_kemiskinan,success,2
3,raw_ipm_sp2010,failed,1
4,raw_ipm_sp2010,skipped,5
5,raw_ipm_sp2010,success,2
6,raw_keparahan_kemiskinan,failed,1
7,raw_keparahan_kemiskinan,skipped,5
8,raw_keparahan_kemiskinan,success,2
9,raw_pengangguran_terbuka,failed,1


,ingest_time,logical_name,stored_filename,source_url,status,notes
35,NaT,raw_garis_kemiskinan,raw_garis_kemiskinan_20260605T155040Z.json,https://data.jabarprov.go.id/api-backend/bigda...,skipped,duplicate_checksum
38,NaT,raw_ipm_sp2010,raw_ipm_sp2010_20260605T155042Z.json,https://data.jabarprov.go.id/api-backend/bigda...,skipped,duplicate_checksum
37,NaT,raw_keparahan_kemiskinan,raw_keparahan_kemiskinan_20260605T155042Z.json,https://data.jabarprov.go.id/api-backend/bigda...,skipped,duplicate_checksum
39,NaT,raw_pengangguran_terbuka,raw_pengangguran_terbuka_20260605T155043Z.json,https://data.jabarprov.go.id/api-backend/bigda...,skipped,duplicate_checksum
36,NaT,raw_persentase_miskin,raw_persentase_miskin_20260605T155041Z.json,https://data.jabarprov.go.id/api-backend/bigda...,skipped,duplicate_checksum


#### 7. Preview Struktur Raw Data

Preview ini bukan tahap cleaning. Tujuannya hanya menunjukkan bahwa output Obtain sudah berupa tabel mentah yang bisa diteruskan ke tahap Scrub. Cleaning, standardisasi, integrasi, dan feature engineering dilakukan pada bagian Scrub di bawah.


In [39]:
for logical_name in extract_module.api_docs:
    csv_path = RAW_DIR / f"{logical_name}.csv"
    print(f"\n=== {logical_name} ===")
    if csv_path.exists():
        display(pd.read_csv(csv_path).head(3))
    else:
        print("File latest belum tersedia. Jalankan extraction terlebih dahulu.")


=== raw_garis_kemiskinan ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,garis_kemiskinan,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,130927,RUPIAH/KAPITA/BULAN,2004
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,111202,RUPIAH/KAPITA/BULAN,2004
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,121902,RUPIAH/KAPITA/BULAN,2004



=== raw_persentase_miskin ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,persentase_penduduk_miskin,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,9.97,PERSEN,2010
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,10.65,PERSEN,2010
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,14.32,PERSEN,2010



=== raw_keparahan_kemiskinan ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,indeks_keparahan_kemiskinan,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,0.62,POIN,2004
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,0.68,POIN,2004
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,0.52,POIN,2004



=== raw_ipm_sp2010 ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,indeks_pembangunan_manusia,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,64.35,POIN,2010
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,60.69,POIN,2010
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,58.58,POIN,2010



=== raw_pengangguran_terbuka ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tingkat_pengangguran_terbuka,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,14.26,PERSEN,2007
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,10.85,PERSEN,2007
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,13.82,PERSEN,2007


#### 8. Proof Orkestrasi

Repository juga menyediakan scaffold Prefect pada `scripts/orchestrate_prefect.py`. Notebook ini tidak menjalankan Prefect secara langsung karena package `prefect` bersifat opsional, tetapi sel berikut mengecek bahwa orchestrator memang mendefinisikan task, flow, dan memanggil script extraction yang sama.


In [40]:
orchestrator_text = ORCHESTRATE_SCRIPT.read_text(encoding="utf-8")
orchestration_checks_df = pd.DataFrame([
    {"check": "File orchestrator tersedia", "status": ORCHESTRATE_SCRIPT.exists()},
    {"check": "Mendefinisikan Prefect task", "status": "@task" in orchestrator_text},
    {"check": "Mendefinisikan Prefect flow", "status": "@flow" in orchestrator_text},
    {"check": "Memanggil scripts/extract_api.py", "status": "scripts/extract_api.py" in orchestrator_text},
    {"check": "Memakai interpreter Python aktif", "status": "sys.executable" in orchestrator_text},
])

orchestration_checks_df

,check,status
0,File orchestrator tersedia,True
1,Mendefinisikan Prefect task,True
2,Mendefinisikan Prefect flow,True
3,Memanggil scripts/extract_api.py,True
4,Memakai interpreter Python aktif,True


#### 9. Ringkasan Obtain

Tahap Obtain sudah menyediakan data mentah dari API Open Data Jabar untuk lima indikator utama. Output disimpan sebagai JSON/CSV raw, dilengkapi metadata endpoint, manifest ingest, checksum, dan scaffold orkestrasi. Dengan struktur ini, tahap Scrub dapat memakai file latest dari `data/raw/`, sementara tim tetap bisa menjalankan ulang extraction jika sumber API diperbarui.


# **2. Scrub**

## *Data Preprocessing Lead* / 18223070

#### 1. Setup

In [41]:
from pathlib import Path
import json
import sys

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "scripts").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

from scripts import preprocess_data as pp

#### 2. Dataset Yang Diproses

Dataset berikut dipakai untuk membangun panel kemiskinan dan kerentanan sosial per kabupaten/kota dan tahun.

In [42]:
dataset_catalog = pd.DataFrame([
    {
        "logical_name": spec.logical_name,
        "metric_col": spec.metric_col,
        "metric_label": spec.metric_label,
        "unit_hint": spec.unit_hint,
        "risk_direction": spec.risk_direction,
    }
    for spec in pp.DATASETS
])

dataset_catalog

,logical_name,metric_col,metric_label,unit_hint,risk_direction
0,raw_garis_kemiskinan,garis_kemiskinan,Garis kemiskinan,rupiah/kapita/bulan,context_only
1,raw_persentase_miskin,persentase_penduduk_miskin,Persentase penduduk miskin,persen,high_is_risky
2,raw_keparahan_kemiskinan,indeks_keparahan_kemiskinan,Indeks keparahan kemiskinan,indeks,high_is_risky
3,raw_ipm_sp2010,indeks_pembangunan_manusia,Indeks pembangunan manusia,indeks,low_is_risky
4,raw_pengangguran_terbuka,tingkat_pengangguran_terbuka,Tingkat pengangguran terbuka,persen,high_is_risky


#### 3. Profiling Raw Data

Bagian ini adalah Explore ringan untuk preprocessing. Tujuannya bukan mencari insight final, tapi memastikan sumber data bisa dibersihkan dan diintegrasikan.

In [43]:
def latest_matching_file(logical_name, suffix):
    candidates = []
    for directory in [pp.DEFAULT_SOURCE_DIR, pp.DEFAULT_ARCHIVE_DIR]:
        candidates.extend(directory.glob(f"{logical_name}{suffix}"))
        candidates.extend(directory.glob(f"{logical_name}_*{suffix}"))
    if not candidates:
        return None
    return sorted(candidates, key=lambda path: (path.stat().st_mtime, path.name), reverse=True)[0]


raw_profiles = []

for spec in pp.DATASETS:
    csv_path = latest_matching_file(spec.logical_name, ".csv")
    json_path = latest_matching_file(spec.logical_name, ".json")

    csv_rows = None
    csv_cols = []
    csv_status = "not_found"
    if csv_path is not None:
        try:
            csv_df = pd.read_csv(csv_path)
            csv_rows = len(csv_df)
            csv_cols = list(csv_df.columns)
            csv_status = "usable_columns" if pp.source_is_usable(csv_df, spec) else "not_statistical_records"
        except Exception as exc:
            csv_status = f"read_error: {exc}"

    json_kind = "not_found"
    json_records = None
    if json_path is not None:
        try:
            with json_path.open("r", encoding="utf-8") as handle:
                payload = json.load(handle)
            if isinstance(payload, dict) and "openapi" in payload:
                json_kind = "openapi_doc"
            else:
                records = pp.extract_records_from_json(payload)
                json_kind = "statistical_records" if records else "unknown_json_shape"
                json_records = len(records)
        except Exception as exc:
            json_kind = f"read_error: {exc}"

    raw_profiles.append({
        "logical_name": spec.logical_name,
        "expected_metric": spec.metric_col,
        "csv_file": str(csv_path.relative_to(PROJECT_ROOT)) if csv_path else "",
        "csv_rows": csv_rows,
        "csv_columns": ", ".join(csv_cols[:8]),
        "csv_status": csv_status,
        "json_file": str(json_path.relative_to(PROJECT_ROOT)) if json_path else "",
        "json_kind": json_kind,
        "json_records": json_records,
    })

raw_profile_df = pd.DataFrame(raw_profiles)
raw_profile_df

,logical_name,expected_metric,csv_file,csv_rows,csv_columns,csv_status,json_file,json_kind,json_records
0,raw_garis_kemiskinan,garis_kemiskinan,data/archive/raw_garis_kemiskinan_20260605T155...,594,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/archive/raw_garis_kemiskinan_20260605T155...,statistical_records,594
1,raw_persentase_miskin,persentase_penduduk_miskin,data/archive/raw_persentase_miskin_20260605T15...,405,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/archive/raw_persentase_miskin_20260605T15...,statistical_records,405
2,raw_keparahan_kemiskinan,indeks_keparahan_kemiskinan,data/archive/raw_keparahan_kemiskinan_20260605...,567,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/archive/raw_keparahan_kemiskinan_20260605...,statistical_records,567
3,raw_ipm_sp2010,indeks_pembangunan_manusia,data/archive/raw_ipm_sp2010_20260605T155042Z.csv,405,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/archive/raw_ipm_sp2010_20260605T155042Z.json,statistical_records,405
4,raw_pengangguran_terbuka,tingkat_pengangguran_terbuka,data/archive/raw_pengangguran_terbuka_20260605...,424,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/archive/raw_pengangguran_terbuka_20260605...,statistical_records,424


#### 4. Expected Schema Check

Untuk integrasi panel, setiap dataset minimal perlu punya key `kode_kabupaten_kota` dan `tahun`, nama wilayah, serta satu kolom metrik utama.

In [44]:
schema_checks = []

for spec in pp.DATASETS:
    csv_path = latest_matching_file(spec.logical_name, ".csv")
    expected_cols = pp.KEY_COLS + ["nama_kabupaten_kota", spec.metric_col]

    if csv_path is None:
        schema_checks.append({
            "logical_name": spec.logical_name,
            "expected_columns": ", ".join(expected_cols),
            "missing_columns": "all",
            "ready_for_scrub": False,
        })
        continue

    raw_df = pd.read_csv(csv_path)
    normalized_cols = set(pp.standardize_columns(raw_df).columns)
    missing_cols = [col for col in expected_cols if col not in normalized_cols]
    schema_checks.append({
        "logical_name": spec.logical_name,
        "expected_columns": ", ".join(expected_cols),
        "missing_columns": ", ".join(missing_cols),
        "ready_for_scrub": len(missing_cols) == 0,
    })

schema_check_df = pd.DataFrame(schema_checks)
schema_check_df

,logical_name,expected_columns,missing_columns,ready_for_scrub
0,raw_garis_kemiskinan,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True
1,raw_persentase_miskin,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True
2,raw_keparahan_kemiskinan,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True
3,raw_ipm_sp2010,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True
4,raw_pengangguran_terbuka,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True


Jika `ready_for_scrub` bernilai `False` dan `json_kind` adalah `openapi_doc`, berarti file raw lokal masih berisi dokumentasi API, bukan record statistik. Notebook ini tetap bisa lanjut kalau internet tersedia dengan mengubah flag `FETCH_OPENAPI_IF_NEEDED` menjadi `True`.

#### 5. Cleaning Per Dataset

Keputusan preprocessing yang digunakan:
- nama kolom distandardisasi ke snake_case
- nama provinsi dan kabupaten/kota distandardisasi ke uppercase
- key `kode_kabupaten_kota` dan `tahun` dipaksa menjadi numerik integer
- metrik dipaksa menjadi numerik
- nilai 0 pada indikator inti diperlakukan sebagai placeholder missing jika tidak masuk akal secara domain
- duplikasi key kabupaten/kota-tahun dikurangi menjadi satu record
- missing value metrik hanya diimputasi untuk gap internal dengan interpolasi per wilayah; missing struktural tetap dibiarkan kosong

In [45]:
FETCH_OPENAPI_IF_NEEDED = False

cleaned_by_metric = {}
quality_profiles = []
source_infos = []
load_errors = []

for spec in pp.DATASETS:
    try:
        raw_df, source_info = pp.load_source_dataframe(
            spec=spec,
            source_dir=pp.DEFAULT_SOURCE_DIR,
            archive_dir=pp.DEFAULT_ARCHIVE_DIR,
            interim_dir=pp.DEFAULT_INTERIM_DIR,
            fetch_openapi_if_needed=FETCH_OPENAPI_IF_NEEDED,
            fetch_limit=5000,
            fetch_timeout=30,
        )
        cleaned_df, profile = pp.clean_dataset(raw_df, spec)
        cleaned_by_metric[spec.metric_col] = cleaned_df
        quality_profiles.append(profile)
        source_infos.append(source_info)
    except Exception as exc:
        load_errors.append({
            "logical_name": spec.logical_name,
            "metric": spec.metric_col,
            "error": str(exc),
        })

quality_profile_df = pd.DataFrame(quality_profiles)
load_error_df = pd.DataFrame(load_errors)

display(quality_profile_df)
display(load_error_df)

,logical_name,metric,rows_raw,rows_clean,missing_values_before,missing_values_after,missing_metric_before_impute,imputed_values,invalid_zero_values,rows_dropped_missing_keys,duplicate_key_rows,exact_duplicate_rows
0,raw_garis_kemiskinan,garis_kemiskinan,594,594,0,14,14,0,14,0,0,0
1,raw_persentase_miskin,persentase_penduduk_miskin,405,405,0,5,5,0,5,0,0,0
2,raw_keparahan_kemiskinan,indeks_keparahan_kemiskinan,567,567,0,14,14,0,14,0,0,0
3,raw_ipm_sp2010,indeks_pembangunan_manusia,405,405,0,3,3,0,3,0,0,0
4,raw_pengangguran_terbuka,tingkat_pengangguran_terbuka,424,424,0,0,0,0,0,0,0,0


""


#### 6. Integrasi Dataset dan Filter Periode

Semua dataset digabung menjadi satu panel dengan key `kode_kabupaten_kota` dan `tahun`, lalu difilter ke periode analisis `2010-2024` sesuai scope obtain.

In [46]:
expected_metric_count = len(pp.DATASETS)

ANALYSIS_START_YEAR = 2010
ANALYSIS_END_YEAR = 2024

if len(cleaned_by_metric) == expected_metric_count:
    panel = pp.integrate_datasets(cleaned_by_metric)
    panel = pp.filter_analysis_period(panel, ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)
    pp.validate_panel(panel)
    print(f"Panel rows: {len(panel):,}")
    print(f"Kabupaten/kota: {panel['kode_kabupaten_kota'].nunique():,}")
    print(f"Year range: {int(panel['tahun'].min())} - {int(panel['tahun'].max())}")
    display(panel.head())
else:
    panel = None
    print("Panel belum bisa dibuat karena belum semua dataset berhasil dibaca dan dibersihkan.")

Panel rows: 405
Kabupaten/kota: 27
Year range: 2010 - 2024


,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tahun,garis_kemiskinan,garis_kemiskinan_was_imputed,persentase_penduduk_miskin,persentase_penduduk_miskin_was_imputed,indeks_keparahan_kemiskinan,indeks_keparahan_kemiskinan_was_imputed,indeks_pembangunan_manusia,indeks_pembangunan_manusia_was_imputed,tingkat_pengangguran_terbuka,tingkat_pengangguran_terbuka_was_imputed
0,32,JAWA BARAT,3201,KABUPATEN BOGOR,2010,214338.0,False,9.97,False,0.43,False,64.35,False,10.64,False
1,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,2010,184127.0,False,10.65,False,0.30,False,60.69,False,9.89,False
2,32,JAWA BARAT,3203,KABUPATEN CIANJUR,2010,202438.0,False,14.32,False,0.38,False,58.58,False,11.21,False
3,32,JAWA BARAT,3204,KABUPATEN BANDUNG,2010,217452.0,False,9.29,False,0.24,False,67.28,False,10.69,False
4,32,JAWA BARAT,3205,KABUPATEN GARUT,2010,180406.0,False,13.94,False,0.34,False,60.23,False,7.75,False


#### 7. Feature Engineering

Fitur yang dibuat:
- perubahan year-over-year tiap metrik
- persentase kemiskinan rolling 3 tahun
- arah tren kemiskinan
- skor kerentanan sosial berbasis z-score tahunan
- peringkat dan bucket prioritas intervensi
- jumlah indikator yang diimputasi

In [47]:
if panel is not None:
    panel_featured = pp.add_features(panel)
    pp.validate_panel(panel_featured)
    display(panel_featured.head(10))
else:
    panel_featured = None
    print("Feature engineering belum dijalankan karena panel belum tersedia.")

,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tahun,garis_kemiskinan,garis_kemiskinan_was_imputed,persentase_penduduk_miskin,persentase_penduduk_miskin_was_imputed,indeks_keparahan_kemiskinan,indeks_keparahan_kemiskinan_was_imputed,indeks_pembangunan_manusia,indeks_pembangunan_manusia_was_imputed,tingkat_pengangguran_terbuka,tingkat_pengangguran_terbuka_was_imputed,garis_kemiskinan_yoy_change,garis_kemiskinan_yoy_pct_change,persentase_penduduk_miskin_yoy_change,persentase_penduduk_miskin_yoy_pct_change,indeks_keparahan_kemiskinan_yoy_change,indeks_keparahan_kemiskinan_yoy_pct_change,indeks_pembangunan_manusia_yoy_change,indeks_pembangunan_manusia_yoy_pct_change,tingkat_pengangguran_terbuka_yoy_change,tingkat_pengangguran_terbuka_yoy_pct_change,persentase_miskin_rolling_3y,arah_tren_kemiskinan,persentase_penduduk_miskin_risk_z,indeks_keparahan_kemiskinan_risk_z,indeks_pembangunan_manusia_risk_z,tingkat_pengangguran_terbuka_risk_z,jumlah_komponen_skor_kerentanan,skor_kerentanan_sosial,peringkat_prioritas_intervensi,prioritas_intervensi,jumlah_indikator_diimputasi,jumlah_indikator_tersedia
0,32,JAWA BARAT,3278,KOTA TASIKMALAYA,2010,263177.0,False,20.71,False,1.17,False,66.58,False,8.16,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.71,belum_ada_pembanding,2.352869,3.504758,-0.057965,-0.723118,4,1.2691,1,sangat_tinggi,0,5
1,32,JAWA BARAT,3215,KABUPATEN KARAWANG,2010,266597.0,False,12.21,False,0.78,False,64.58,False,14.88,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.21,belum_ada_pembanding,0.211695,1.595126,0.324059,1.492651,4,0.9059,2,sangat_tinggi,0,5
2,32,JAWA BARAT,3217,KABUPATEN BANDUNG BARAT,2010,216388.0,False,14.68,False,0.58,False,61.34,False,13.31,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.68,belum_ada_pembanding,0.833895,0.615828,0.942938,0.974979,4,0.8419,3,sangat_tinggi,0,5
3,32,JAWA BARAT,3209,KABUPATEN CIREBON,2010,230346.0,False,16.12,False,0.58,False,63.64,False,12.97,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.12,belum_ada_pembanding,1.196635,0.615828,0.503611,0.862871,4,0.7947,4,sangat_tinggi,0,5
4,32,JAWA BARAT,3212,KABUPATEN INDRAMAYU,2010,264576.0,False,16.58,False,0.52,False,60.86,False,11.29,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.58,belum_ada_pembanding,1.312510,0.322038,1.034624,0.308929,4,0.7445,5,sangat_tinggi,0,5
5,32,JAWA BARAT,3203,KABUPATEN CIANJUR,2010,202438.0,False,14.32,False,0.38,False,58.58,False,11.21,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.32,belum_ada_pembanding,0.743210,-0.363470,1.470132,0.282551,4,0.5331,6,tinggi,0,5
6,32,JAWA BARAT,3210,KABUPATEN MAJALENGKA,2010,263377.0,False,15.51,False,0.67,False,62.30,False,5.82,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.51,belum_ada_pembanding,1.042974,1.056512,0.759567,-1.494680,4,0.3411,7,tinggi,0,5
7,32,JAWA BARAT,3272,KOTA SUKABUMI,2010,284339.0,False,9.24,False,0.52,False,67.94,False,15.65,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.24,belum_ada_pembanding,-0.536456,0.322038,-0.317741,1.746541,4,0.3036,8,tinggi,0,5
8,32,JAWA BARAT,3271,KOTA BOGOR,2010,278530.0,False,9.47,False,0.48,False,71.25,False,17.20,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.47,belum_ada_pembanding,-0.478518,0.126179,-0.949991,2.257619,4,0.2388,9,tinggi,0,5
9,32,JAWA BARAT,3213,KABUPATEN SUBANG,2010,234803.0,False,13.54,False,0.52,False,63.54,False,8.72,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.54,belum_ada_pembanding,0.546726,0.322038,0.522712,-0.538470,4,0.2133,10,tinggi,0,5


#### 8. Validasi Output Preprocessing

Validasi ini memastikan output tidak punya duplikasi key dan menunjukkan missing value yang masih tersisa setelah preprocessing.

In [48]:
if panel_featured is not None:
    validation_summary = {
        "rows": len(panel_featured),
        "duplicate_kabupaten_kota_year_keys": int(panel_featured.duplicated(pp.KEY_COLS).sum()),
        "missing_key_values": int(panel_featured[pp.KEY_COLS].isna().sum().sum()),
        "kabupaten_kota_count": int(panel_featured["kode_kabupaten_kota"].nunique()),
        "year_min": int(panel_featured["tahun"].min()),
        "year_max": int(panel_featured["tahun"].max()),
    }
    display(pd.DataFrame([validation_summary]))
    display(panel_featured.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(20))
else:
    print("Validasi output belum bisa dijalankan karena panel final belum tersedia.")

,rows,duplicate_kabupaten_kota_year_keys,missing_key_values,kabupaten_kota_count,year_min,year_max
0,405,0,0,27,2010,2024


,missing_count
tingkat_pengangguran_terbuka_yoy_change,86
tingkat_pengangguran_terbuka_yoy_pct_change,86
indeks_keparahan_kemiskinan_yoy_pct_change,32
indeks_keparahan_kemiskinan_yoy_change,32
garis_kemiskinan_yoy_pct_change,32
tingkat_pengangguran_terbuka_risk_z,32
skor_kerentanan_sosial,32
prioritas_intervensi,32
peringkat_prioritas_intervensi,32
tingkat_pengangguran_terbuka_was_imputed,32


#### 9. Preview Prioritas Intervensi

Preview ini bukan pengganti analisis akhir, tapi membantu memastikan fitur prioritas sudah terbentuk dan masuk akal secara struktur.

In [49]:
if panel_featured is not None:
    scored_years = panel_featured.loc[panel_featured["skor_kerentanan_sosial"].notna(), "tahun"]
    latest_year = scored_years.max() if not scored_years.empty else panel_featured["tahun"].max()
    preview_cols = [
        "tahun",
        "peringkat_prioritas_intervensi",
        "nama_kabupaten_kota",
        "skor_kerentanan_sosial",
        "prioritas_intervensi",
        "persentase_penduduk_miskin",
        "indeks_keparahan_kemiskinan",
        "tingkat_pengangguran_terbuka",
        "indeks_pembangunan_manusia",
    ]
    display(
        panel_featured.loc[
            panel_featured["tahun"].eq(latest_year) & panel_featured["skor_kerentanan_sosial"].notna(),
            preview_cols,
        ]
        .sort_values("peringkat_prioritas_intervensi")
        .head(10)
    )
else:
    print("Preview prioritas belum tersedia karena panel final belum tersedia.")

,tahun,peringkat_prioritas_intervensi,nama_kabupaten_kota,skor_kerentanan_sosial,prioritas_intervensi,persentase_penduduk_miskin,indeks_keparahan_kemiskinan,tingkat_pengangguran_terbuka,indeks_pembangunan_manusia
378,2024,1,KABUPATEN KUNINGAN,1.2018,sangat_tinggi,11.88,0.53,7.78,71.26
379,2024,2,KABUPATEN INDRAMAYU,1.0761,sangat_tinggi,11.93,0.54,6.25,69.83
380,2024,3,KABUPATEN CIANJUR,0.7443,sangat_tinggi,10.14,0.41,5.99,67.24
381,2024,4,KABUPATEN SUBANG,0.6687,sangat_tinggi,9.49,0.46,6.73,71.36
382,2024,5,KABUPATEN CIREBON,0.6097,sangat_tinggi,11.00,0.36,6.74,71.44
383,2024,6,KABUPATEN GARUT,0.5188,tinggi,9.68,0.29,6.96,68.79
384,2024,7,KABUPATEN MAJALENGKA,0.4586,tinggi,10.82,0.45,4.01,69.74
385,2024,8,KABUPATEN SUMEDANG,0.3969,tinggi,9.10,0.45,6.16,73.73
386,2024,9,KABUPATEN BANDUNG BARAT,0.3696,tinggi,10.49,0.23,6.70,70.03
387,2024,10,KABUPATEN PURWAKARTA,0.3641,tinggi,8.41,0.35,7.34,72.65


#### 10. Simpan Output

Default cell ini dibuat **read-only** untuk mode presentasi notebook. Output preprocessing hanya akan ditulis ulang ke `data/processed` dan `reports/preprocessing` jika `WRITE_OUTPUTS` diubah menjadi `True`.


In [50]:
WRITE_OUTPUTS = False

if WRITE_OUTPUTS and panel_featured is not None:
    pp.DEFAULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pp.DEFAULT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

    panel_path = pp.DEFAULT_OUTPUT_DIR / "panel_kemiskinan_jabar_preprocessed.csv"
    dictionary_path = pp.DEFAULT_OUTPUT_DIR / "feature_dictionary.csv"

    panel_featured.to_csv(panel_path, index=False)
    pd.DataFrame(pp.feature_dictionary_rows()).to_csv(dictionary_path, index=False)
    manifest_path = pp.append_manifest(pp.DEFAULT_OUTPUT_DIR, panel_path, source_infos, panel_featured)
    summary_path = pp.write_quality_outputs(quality_profiles, source_infos, pp.DEFAULT_REPORT_DIR)

    print("Output saved:")
    for path in [panel_path, dictionary_path, manifest_path, summary_path]:
        print(f"- {path.relative_to(PROJECT_ROOT)}")
elif panel_featured is None:
    print("Output not saved, final panel is not ready")
else:
    print("WRITE_OUTPUTS is False, output not saved")

WRITE_OUTPUTS is False, output not saved


# 3. *Explore*

## *Data Analyst / Modeler* / 18222130

Stage E (Explore) and stage M (Model) belong to the Data Analyst / Modeler This section computes the exploratory findings; chart rendering is handed to the Visualization / Dashboard Developer, and the final insight and recommendation narrative is handed to the Documentation and Insight Lead

Scope: the analysis unit is kabupaten/kota The desa level named in the project title is not available from the obtained sources and is out of scope

#### 1. Setup

In [51]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "scripts").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

from scripts import explore_analysis as ea
from scripts import model_clustering as mc

panel = ea.load_panel(ea.DEFAULT_PANEL_PATH)
snapshot_year = ea.latest_scored_year(panel)
snapshot = ea.cross_section(panel, snapshot_year)
print(f"Panel rows: {len(panel)} | regions: {panel['kode_kabupaten_kota'].nunique()} | snapshot year: {snapshot_year}")

Panel rows: 405 | regions: 27 | snapshot year: 2024


#### 2 Descriptive Statistics

Summary of the main indicators in the snapshot year

In [52]:
descriptive = ea.descriptive_stats(snapshot)
display(descriptive.round(3))

,indicator,label,count,mean,std,min,25%,50%,75%,max
0,garis_kemiskinan,Poverty line (rupiah/capita/month),27.0,536065.111,125444.280,393464.000,444743.000,475046.000,604956.000,843893.000
1,persentase_penduduk_miskin,Poverty rate (percent),27.0,8.014,2.649,2.340,6.360,8.410,10.185,11.930
2,indeks_keparahan_kemiskinan,Poverty severity index,27.0,0.280,0.126,0.070,0.200,0.260,0.355,0.540
3,indeks_pembangunan_manusia,Human development index,27.0,73.814,4.681,67.240,69.940,72.650,76.645,83.540
4,tingkat_pengangguran_terbuka,Open unemployment rate (percent),27.0,6.525,1.707,1.580,6.205,6.730,7.590,8.970
5,skor_kerentanan_sosial,skor_kerentanan_sosial,27.0,-0.000,0.641,-1.502,-0.386,0.066,0.428,1.202


#### 3 Correlation and Collinearity

The poverty line correlates strongly with the human development index, so it is excluded from the clustering distance and kept only as economic context

In [53]:
correlation = ea.correlation_matrix(snapshot)
display(correlation.round(3))
display(pd.DataFrame(ea.high_correlation_pairs(snapshot)))

,indicator,garis_kemiskinan,persentase_penduduk_miskin,indeks_keparahan_kemiskinan,indeks_pembangunan_manusia,tingkat_pengangguran_terbuka
0,garis_kemiskinan,1.000,-0.631,-0.405,0.812,0.395
1,persentase_penduduk_miskin,-0.631,1.000,0.719,-0.781,-0.316
2,indeks_keparahan_kemiskinan,-0.405,0.719,1.000,-0.520,-0.091
3,indeks_pembangunan_manusia,0.812,-0.781,-0.520,1.000,0.446
4,tingkat_pengangguran_terbuka,0.395,-0.316,-0.091,0.446,1.000


,indicator_a,indicator_b,pearson_r
0,garis_kemiskinan,indeks_pembangunan_manusia,0.8122


#### 4 Province Trend and Region Change

Province-wide yearly means and the earliest-to-latest poverty change per region

In [54]:
trend = ea.poverty_trend_by_year(panel)
change = ea.region_change_summary(panel)
display(trend)
display(change.head(10))

,tahun,garis_kemiskinan,persentase_penduduk_miskin,indeks_keparahan_kemiskinan,indeks_pembangunan_manusia,tingkat_pengangguran_terbuka,regions_with_score
0,2010,241252.0769,11.3696,0.4542,66.2765,10.3531,26.0
1,2011,266768.2308,10.9873,0.3919,66.8773,9.5888,26.0
2,2012,287490.4615,10.4008,0.3869,67.5119,8.9319,26.0
3,2013,309083.6923,9.9869,0.3346,68.1515,9.0723,26.0
4,2014,320446.5000,9.5500,0.3058,68.5593,8.4546,26.0
5,2015,335103.2593,10.0152,0.4019,69.1763,8.5626,27.0
6,2016,354271.1852,9.4356,0.3496,69.6778,NaN,NaN
7,2017,372047.5556,9.2296,0.3563,70.2789,7.8956,27.0
8,2018,398871.5185,7.9419,0.2881,70.9700,7.9300,27.0
9,2019,413142.5185,7.4067,0.2115,71.6370,7.8489,27.0


,kode_kabupaten_kota,nama_kabupaten_kota,tahun_awal,tahun_akhir,persentase_miskin_awal,persentase_miskin_akhir,perubahan_poin
0,3278,KOTA TASIKMALAYA,2010,2024,20.71,11.10,-9.61
1,3209,KABUPATEN CIREBON,2010,2024,16.12,11.00,-5.12
2,3210,KABUPATEN MAJALENGKA,2010,2024,15.51,10.82,-4.69
3,3212,KABUPATEN INDRAMAYU,2010,2024,16.58,11.93,-4.65
4,3215,KABUPATEN KARAWANG,2010,2024,12.21,7.86,-4.35
5,3205,KABUPATEN GARUT,2010,2024,13.94,9.68,-4.26
6,3217,KABUPATEN BANDUNG BARAT,2010,2024,14.68,10.49,-4.19
7,3203,KABUPATEN CIANJUR,2010,2024,14.32,10.14,-4.18
8,3213,KABUPATEN SUBANG,2010,2024,13.54,9.49,-4.05
9,3211,KABUPATEN SUMEDANG,2010,2024,12.94,9.10,-3.84


#### 5 Administrative-Type Comparison and Extremes

The kota versus kabupaten split is a crude administrative cut, not an urban-industrial versus rural-agrarian split; the model section derives the structural archetypes

In [55]:
display(ea.kota_vs_kabupaten(snapshot))
display(ea.top_bottom_regions(snapshot).round(3))

,tipe_administratif,garis_kemiskinan,persentase_penduduk_miskin,indeks_keparahan_kemiskinan,indeks_pembangunan_manusia,tingkat_pengangguran_terbuka,jumlah_wilayah
0,kabupaten,479999.2222,9.0044,0.3172,71.2117,6.1678,18
1,kota,648196.8889,6.0344,0.2056,79.0200,7.2389,9


,nama_kabupaten_kota,skor_kerentanan_sosial,persentase_penduduk_miskin,indeks_pembangunan_manusia,kelompok
0,KABUPATEN KUNINGAN,1.202,11.88,71.26,paling_rentan
1,KABUPATEN INDRAMAYU,1.076,11.93,69.83,paling_rentan
2,KABUPATEN CIANJUR,0.744,10.14,67.24,paling_rentan
3,KABUPATEN SUBANG,0.669,9.49,71.36,paling_rentan
4,KABUPATEN CIREBON,0.610,11.00,71.44,paling_rentan
5,KABUPATEN BANDUNG,-0.608,6.19,74.27,paling_tidak_rentan
6,KABUPATEN CIAMIS,-0.624,7.39,72.56,paling_tidak_rentan
7,KOTA BEKASI,-0.862,4.01,83.54,paling_tidak_rentan
8,KOTA BANDUNG,-1.078,3.87,83.52,paling_tidak_rentan
9,KOTA DEPOK,-1.502,2.34,82.91,paling_tidak_rentan


#### 6 Visualization Handoff

The Visualization / Dashboard Developer renders charts from the tables saved by `scripts/explore_analysis.py` under `reports/exploration/`:

- `descriptive_stats_latest.csv` for indicator distributions
- `correlation_matrix_latest.csv` for the correlation heatmap
- `poverty_trend_by_year.csv` for the province trend line
- `region_change_summary.csv` for per-region change bars
- `top_bottom_regions_latest.csv` for the most and least vulnerable regions

No chart code is placed here to keep the analyst and visualization roles separate

#### 7 Exploration Findings

- Poverty rate, severity, and the human development index move together across regions, so a small set of indicators already separates the province (supports RQ1)
- The poverty line and unemployment rate behave differently from the deprivation indicators, signalling a distinct urban-industrial profile (supports RQ2)
- Province-wide poverty fell over the panel but the recent path differs by region, which the priority matrix uses later (supports RQ3)

In [56]:
RUN_EXPLORE_EXPORT = False

if RUN_EXPLORE_EXPORT:
    from types import SimpleNamespace
    ea.run(SimpleNamespace(panel_path=str(ea.DEFAULT_PANEL_PATH), report_dir=str(ea.DEFAULT_REPORT_DIR), year=None))
    print("Exploration outputs written to reports/exploration/")
else:
    print("RUN_EXPLORE_EXPORT = False; tables above are not re-exported")

RUN_EXPLORE_EXPORT = False; tables above are not re-exported


# 4. *Model*

#### 1 Build Modeling Frame

The model clusters the snapshot year and attaches a poverty-trend slope per region

In [57]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

model_year = mc.resolve_model_year(panel, None)
model_frame = mc.build_model_frame(panel, model_year, mc.TREND_WINDOW)
print(f"Model year: {model_year} | regions: {len(model_frame)}")
print(f"Clustering features: {mc.CLUSTER_FEATURES}")
display(model_frame[["nama_kabupaten_kota"] + mc.CLUSTER_FEATURES + ["tren_kemiskinan"]].head())

Model year: 2024 | regions: 27
Clustering features: ['persentase_penduduk_miskin', 'indeks_keparahan_kemiskinan', 'indeks_pembangunan_manusia', 'tingkat_pengangguran_terbuka']


,nama_kabupaten_kota,persentase_penduduk_miskin,indeks_keparahan_kemiskinan,indeks_pembangunan_manusia,tingkat_pengangguran_terbuka,tren_kemiskinan
0,KABUPATEN KUNINGAN,11.88,0.53,71.26,7.78,stabil
1,KABUPATEN INDRAMAYU,11.93,0.54,69.83,6.25,memburuk
2,KABUPATEN CIANJUR,10.14,0.41,67.24,5.99,memburuk
3,KABUPATEN SUBANG,9.49,0.46,71.36,6.73,memburuk
4,KABUPATEN CIREBON,11.00,0.36,71.44,6.74,memburuk


#### 2 Standardize and Select Cluster Count

Indicators are standardized; the table reports validation metrics across candidate k k = 3 is chosen for policy legibility

In [58]:
scaler = StandardScaler()
scaled = scaler.fit_transform(model_frame[mc.CLUSTER_FEATURES])
scaled_df = pd.DataFrame(scaled, columns=mc.CLUSTER_FEATURES, index=model_frame.index)

selection = mc.select_k(scaled, mc.DEFAULT_K_MAX)
display(selection)

K = mc.DEFAULT_K
print(f"Selected k = {K}")

,k,silhouette,calinski_harabasz,davies_bouldin,inertia
0,2,0.3843,21.4453,0.8845,58.1329
1,3,0.2932,18.1364,1.1658,43.0045
2,4,0.3351,20.6145,0.9333,29.2774
3,5,0.3044,19.4960,0.9477,23.7638
4,6,0.2633,17.8305,1.0218,20.5896


Selected k = 3


#### 3 Fit Clustering and Stability Check

K-Means clusters are relabelled by descending vulnerability, then cross-checked against Agglomerative (Ward) clustering

In [59]:
kmeans = KMeans(n_clusters=K, n_init=10, random_state=mc.RANDOM_STATE)
raw_labels = kmeans.fit_predict(scaled)
model_frame["cluster_id"] = mc.order_labels_by_vulnerability(model_frame, raw_labels)

ward_labels = AgglomerativeClustering(n_clusters=K, linkage="ward").fit_predict(scaled)
ari = adjusted_rand_score(raw_labels, ward_labels)
silhouette = silhouette_score(scaled, raw_labels)
print(f"Silhouette: {silhouette:.4f} | KMeans vs Ward ARI: {ari:.4f}")

components = PCA(n_components=2, random_state=mc.RANDOM_STATE).fit_transform(scaled)
model_frame["pc1"], model_frame["pc2"] = components[:, 0].round(4), components[:, 1].round(4)

Silhouette: 0.2932 | KMeans vs Ward ARI: 0.7607


#### 4 Cluster Profiles and Archetypes (RQ1)

Each cluster is summarized on the original scale and labelled with an economic archetype derived from its indicator profile

In [60]:
profiles, tier_map, profile_map = mc.profile_clusters(model_frame, K)
model_frame["cluster_vulnerability_tier"] = model_frame["cluster_id"].map(tier_map)
model_frame["profil_ekonomi"] = model_frame["cluster_id"].map(profile_map)
display(profiles.drop(columns="anggota"))
for _, row in profiles.iterrows():
    print(f"Cluster {row['cluster_id']} ({row['cluster_vulnerability_tier']}, {row['profil_ekonomi']}): {row['anggota']}")

,cluster_id,cluster_vulnerability_tier,profil_ekonomi,jumlah_wilayah,jumlah_kota,jumlah_kabupaten,mean_persentase_penduduk_miskin,mean_indeks_keparahan_kemiskinan,mean_indeks_pembangunan_manusia,mean_tingkat_pengangguran_terbuka,mean_garis_kemiskinan,mean_skor_kerentanan_sosial
0,0,kerentanan_tinggi,rural_agraris,7,0,7,10.6229,0.4571,70.6571,6.2371,475338.2857,0.7366
1,1,kerentanan_menengah,campuran,12,3,9,8.5583,0.2542,71.8550,5.8667,476968.0833,0.0087
2,2,kerentanan_rendah,urban_industri,8,6,2,4.9162,0.1638,79.5162,7.7637,677846.6250,-0.6575


Cluster 0 (kerentanan_tinggi, rural_agraris): KABUPATEN CIANJUR; KABUPATEN CIREBON; KABUPATEN INDRAMAYU; KABUPATEN KUNINGAN; KABUPATEN MAJALENGKA; KABUPATEN SUBANG; KABUPATEN SUMEDANG
Cluster 1 (kerentanan_menengah, campuran): KABUPATEN BANDUNG BARAT; KABUPATEN BOGOR; KABUPATEN CIAMIS; KABUPATEN GARUT; KABUPATEN KARAWANG; KABUPATEN PANGANDARAN; KABUPATEN PURWAKARTA; KABUPATEN SUKABUMI; KABUPATEN TASIKMALAYA; KOTA BANJAR; KOTA CIREBON; KOTA TASIKMALAYA
Cluster 2 (kerentanan_rendah, urban_industri): KABUPATEN BANDUNG; KABUPATEN BEKASI; KOTA BANDUNG; KOTA BEKASI; KOTA BOGOR; KOTA CIMAHI; KOTA DEPOK; KOTA SUKABUMI


#### 5 Feature Importance (RQ2)

Eta squared ranks how strongly each indicator separates the clusters; Cohen's d contrasts the urban-industrial and rural-agrarian archetypes

In [61]:
importance = mc.feature_importance(model_frame, scaled_df)
comparison = mc.urban_rural_comparison(model_frame)
display(importance)
display(comparison)

,feature,label,eta_squared,anova_f,anova_p
0,indeks_keparahan_kemiskinan,Poverty severity index,0.8072,50.2273,0.000000
1,persentase_penduduk_miskin,Poverty rate,0.7016,28.2174,0.000000
2,indeks_pembangunan_manusia,Human development index,0.6600,23.2923,0.000002
3,tingkat_pengangguran_terbuka,Open unemployment rate,0.2385,3.7580,0.038034


,feature,label,mean_urban_industri,mean_rural_agraris,cohens_d,welch_p
0,indeks_keparahan_kemiskinan,Poverty severity index,0.1638,0.4571,-4.7725,0.000001
1,persentase_penduduk_miskin,Poverty rate,4.9162,10.6229,-4.0673,0.000003
2,indeks_pembangunan_manusia,Human development index,79.5162,70.6571,3.0014,0.000079
3,garis_kemiskinan,Poverty line,677846.6250,475338.2857,2.0028,0.002282
4,tingkat_pengangguran_terbuka,Open unemployment rate,7.7637,6.2371,1.4082,0.019084


#### 6 Priority Matrix (RQ3)

The matrix crosses the current vulnerability level with the poverty trend direction; the worsening high-vulnerability cell is the fastest-intervention group

In [62]:
model_frame = mc.assign_priority(model_frame)
matrix = mc.build_priority_matrix(model_frame)
display(matrix)

validity = mc.face_validity(model_frame)
print(f"Face-validity: {'PASS' if validity['passed'] else 'REVIEW'}")
print(f"Expected high-vulnerability in top tier: {validity['expected_high_in_top_tier']}")
print(f"Expected low-vulnerability in bottom tier: {validity['expected_low_in_bottom_tier']}")

,level_kerentanan,tren_kemiskinan,prioritas_aksi,jumlah_wilayah,dominant_cluster_id,wilayah
0,tinggi,memburuk,prioritas_segera,7,0,KABUPATEN BANDUNG BARAT; KABUPATEN CIANJUR; KA...
1,tinggi,stabil,intervensi_intensif,1,0,KABUPATEN KUNINGAN
2,tinggi,membaik,intervensi_intensif,1,0,KABUPATEN SUMEDANG
3,sedang,memburuk,peringatan_dini,6,1,KABUPATEN PURWAKARTA; KABUPATEN SUKABUMI; KABU...
4,sedang,stabil,pemantauan,2,1,KABUPATEN BOGOR; KABUPATEN KARAWANG
5,sedang,membaik,pemantauan,1,1,KOTA TASIKMALAYA
6,rendah,memburuk,pantau,4,1,KABUPATEN BEKASI; KABUPATEN CIAMIS; KABUPATEN ...
7,rendah,stabil,rutin,5,2,KABUPATEN BANDUNG; KOTA BANJAR; KOTA BEKASI; K...


Face-validity: PASS
Expected high-vulnerability in top tier: ['KABUPATEN CIANJUR', 'KABUPATEN KUNINGAN']
Expected low-vulnerability in bottom tier: ['KOTA BEKASI', 'KOTA DEPOK', 'KOTA BANDUNG', 'KABUPATEN BEKASI']


#### 7 Visualization Handoff

The Visualization / Dashboard Developer renders charts from `data/processed/region_clusters_{year}.csv` and the tables under `reports/modeling/`:

- `pc1`, `pc2` with `cluster_id` for the cluster scatter
- `cluster_profiles.csv` for the archetype profile bars
- `feature_importance.csv` for the separator ranking
- `priority_matrix.csv` for the level-by-trend matrix and the region priority map

No chart code is placed here

#### 8 Persist Model Outputs

In [63]:
RUN_MODEL_PIPELINE = False

if RUN_MODEL_PIPELINE:
    from types import SimpleNamespace
    outputs = mc.run(SimpleNamespace(
        panel_path=str(mc.DEFAULT_PANEL_PATH),
        output_dir=str(mc.DEFAULT_OUTPUT_DIR),
        report_dir=str(mc.DEFAULT_REPORT_DIR),
        year=None,
        k=mc.DEFAULT_K,
        k_max=mc.DEFAULT_K_MAX,
        trend_window=mc.TREND_WINDOW,
    ))
    for name, path in outputs.items():
        print(f"- {name}: {path.relative_to(PROJECT_ROOT)}")
else:
    print("RUN_MODEL_PIPELINE = False; outputs not rewritten Run scripts/model_clustering.py to regenerate")

RUN_MODEL_PIPELINE = False; outputs not rewritten Run scripts/model_clustering.py to regenerate


#### 9 Model summary

This maps the model to the report's model section so the Documentation Lead can lift it directly

| Component | Detail |
| --- | --- |
| Goal | Group West Java kabupaten/kota by poverty and vulnerability profile so help is targeted per group, not spread evenly |
| Method | K-Means on four standardized indicators, checked against Agglomerative (Ward) |
| Input variables | Poverty rate, poverty severity, human development index, open unemployment The poverty line is kept as context, not in the distance, because it tracks the development index too closely |
| Output | Per region: cluster, vulnerability tier, economic archetype, poverty trend, and a priority action |
| Evaluation metrics | Silhouette, Calinski-Harabasz, Davies-Bouldin, and the K-Means versus Ward adjusted Rand index |
| Evaluation result | The values are printed in the selection and fit cells above; the two algorithms agree on most regions, so the grouping holds |
| Limitations | Only 27 regions, a single snapshot year, and a trend direction that depends on the window chosen |

#### 10 Region output columns

The columns added in `region_clusters_{year}.csv`, for whoever reads it next:

- `cluster_id`: cluster ordered by vulnerability, 0 is the most vulnerable
- `cluster_vulnerability_tier`: readable tier name for that cluster
- `profil_ekonomi`: archetype from the indicator tertiles, urban_industri, rural_agraris, or campuran
- `tipe_administratif`: kota or kabupaten, a rough administrative label only
- `pc1`, `pc2`: the two principal components of the scaled features, for plotting
- `tren_slope_poin_per_tahun` and `tren_kemiskinan`: the poverty slope and its direction
- `level_kerentanan`: vulnerability tertile within the snapshot year
- `prioritas_aksi`: the action class from the level by trend matrix

# 5. *iNterpret*

This section holds the analyst's technical interpretation of the model output The minimum three insights and three recommendations required by the rubric are written by the Documentation and Insight Lead, who uses the tables below as the evidence base

#### 1 Evidence Base

The region table is taken from the in-memory `model_frame` built in the Model section, so this section runs without re-reading any file

In [64]:
priority_first = model_frame.loc[model_frame["prioritas_aksi"].eq("prioritas_segera")]
display(priority_first[["nama_kabupaten_kota", "cluster_vulnerability_tier", "profil_ekonomi", "skor_kerentanan_sosial", "tren_kemiskinan", "prioritas_aksi"]])
display(importance[["feature", "label", "eta_squared"]])

,nama_kabupaten_kota,cluster_vulnerability_tier,profil_ekonomi,skor_kerentanan_sosial,tren_kemiskinan,prioritas_aksi
1,KABUPATEN INDRAMAYU,kerentanan_tinggi,rural_agraris,1.0761,memburuk,prioritas_segera
2,KABUPATEN CIANJUR,kerentanan_tinggi,rural_agraris,0.7443,memburuk,prioritas_segera
3,KABUPATEN SUBANG,kerentanan_tinggi,rural_agraris,0.6687,memburuk,prioritas_segera
4,KABUPATEN CIREBON,kerentanan_tinggi,rural_agraris,0.6097,memburuk,prioritas_segera
5,KABUPATEN GARUT,kerentanan_menengah,campuran,0.5188,memburuk,prioritas_segera
6,KABUPATEN MAJALENGKA,kerentanan_tinggi,rural_agraris,0.4586,memburuk,prioritas_segera
8,KABUPATEN BANDUNG BARAT,kerentanan_menengah,campuran,0.3696,memburuk,prioritas_segera


,feature,label,eta_squared
0,indeks_keparahan_kemiskinan,Poverty severity index,0.8072
1,persentase_penduduk_miskin,Poverty rate,0.7016
2,indeks_pembangunan_manusia,Human development index,0.6600
3,tingkat_pengangguran_terbuka,Open unemployment rate,0.2385


#### 2 Technical findings (handoff to the Documentation and Insight Lead)

These are readings of the tables above Turning them into the required insights and recommendations is the Documentation and Insight Lead's job

- RQ1: the regions fall into three groups, a high-vulnerability rural-agrarian cluster, a mixed middle cluster, and a low-vulnerability urban-industrial cluster K-Means and Ward agree on most regions, so the split is stable rather than an artefact of one algorithm
- RQ2: poverty severity and the poverty rate separate the clusters the most by eta squared The human development index and the poverty line are what mark the urban-industrial group, and open unemployment runs higher there, not lower, which fits an industrial labour market
- RQ3: the regions in the high-vulnerability and worsening cell of the matrix are the ones to act on first The full ranked list is in data/processed/region_clusters_2024.csv

Two caveats worth stating plainly The top cluster is mostly the north-coast belt (Indramayu, Kuningan, Cirebon, Majalengka) plus Cianjur; Garut and Tasikmalaya land in the middle cluster because severity and the other indicators, not the headline poverty rate, decide membership That sharpens the background assumption instead of breaking it Also, the cluster and the vulnerability score are built from the same indicators, so they agree by construction; the trend axis is what adds independent signal, and since it moves with the window chosen, it is anchored to the last pre-pandemic year